# PHASE 6 — THREAT RESPONSE & AUTOMATED RESPONSE SIMULATION

## Step 6.0 — Environment & Dependency Verification

In [2]:
import sys

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

Python executable:
d:\smart-cybersecurity-ml\.venv\Scripts\python.exe

Python version:
3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]


Step 6.0.1 — Check required packages

In [3]:
import numpy
import pandas
import sklearn
import xgboost
import joblib
import matplotlib

print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("Scikit-learn:", sklearn.__version__)
print("XGBoost:", xgboost.__version__)
print("Joblib:", joblib.__version__)
print("Matplotlib:", matplotlib.__version__)

print("\nAll Phase 6 core packages are available.")

NumPy: 2.5.2
Pandas: 3.0.5
Scikit-learn: 1.9.0
XGBoost: 3.4.1
Joblib: 1.6.0
Matplotlib: 3.11.1

All Phase 6 core packages are available.


## Step 6.1 — Load Trained Model and Preprocessing Objects

In [4]:
import os
import joblib
import numpy as np
import pandas as pd

PROJECT_ROOT = r"D:\smart-cybersecurity-ml"

MODEL_PATH = os.path.join(
    PROJECT_ROOT,
    "models",
    "final_xgboost_model.pkl"
)

SCALER_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "scaler.pkl"
)

TARGET_ENCODER_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "target_encoder.pkl"
)

X_TEST_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "X_test_encoded.csv"
)

Y_TEST_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "y_test_encoded.npy"
)

PHASE6_RESULTS = os.path.join(
    PROJECT_ROOT,
    "results",
    "phase6"
)

os.makedirs(PHASE6_RESULTS, exist_ok=True)

print("Paths configured successfully.")
print("Phase 6 results folder:")
print(PHASE6_RESULTS)

Paths configured successfully.
Phase 6 results folder:
D:\smart-cybersecurity-ml\results\phase6


Load all objects

In [5]:
xgb_model = joblib.load(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)
target_encoder = joblib.load(TARGET_ENCODER_PATH)

X_test = pd.read_csv(X_TEST_PATH)
y_test = np.load(Y_TEST_PATH)

print("Model loaded successfully.")
print("Scaler loaded successfully.")
print("Target encoder loaded successfully.")

print("\nX_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

Model loaded successfully.
Scaler loaded successfully.
Target encoder loaded successfully.

X_test shape: (29696, 41)
y_test shape: (29696,)


## Step 6.2 — Attack Severity Mapping

In [6]:
SEVERITY_MAPPING = {
    # Normal traffic
    "normal": "Low",

    # Low severity attacks
    "ipsweep": "Low",
    "portsweep": "Low",
    "nmap": "Low",

    # Medium severity attacks
    "satan": "Medium",
    "probe": "Medium",
    "warezclient": "Medium",
    "warezmaster": "Medium",
    "ftp_write": "Medium",
    "imap": "Medium",
    "multihop": "Medium",
    "phf": "Medium",
    "spy": "Medium",
    "loadmodule": "Medium",

    # High severity attacks
    "neptune": "High",
    "back": "High",
    "land": "High",
    "pod": "High",
    "teardrop": "High",
    "smurf": "High",
    "guess_passwd": "High",
    "buffer_overflow": "High",

    # Critical severity attacks
    "rootkit": "Critical",
    "perl": "Critical",
    "sqlattack": "Critical",
    "xterm": "Critical"
}

print("Severity mapping created.")
print("Total mapped attack classes:", len(SEVERITY_MAPPING))

for severity in ["Low", "Medium", "High", "Critical"]:
    attacks = [
        attack
        for attack, level in SEVERITY_MAPPING.items()
        if level == severity
    ]

    print(f"\n{severity}:")
    print(attacks)

Severity mapping created.
Total mapped attack classes: 26

Low:
['normal', 'ipsweep', 'portsweep', 'nmap']

Medium:
['satan', 'probe', 'warezclient', 'warezmaster', 'ftp_write', 'imap', 'multihop', 'phf', 'spy', 'loadmodule']

High:
['neptune', 'back', 'land', 'pod', 'teardrop', 'smurf', 'guess_passwd', 'buffer_overflow']

Critical:
['rootkit', 'perl', 'sqlattack', 'xterm']


## Step 6.3 — Automated Response Mapping

In [7]:
RESPONSE_MAPPING = {
    "Low": {
        "action": "Monitor",
        "description": "Continue monitoring the traffic and record the event."
    },

    "Medium": {
        "action": "Alert",
        "description": "Generate an alert and increase monitoring."
    },

    "High": {
        "action": "Investigate",
        "description": "Trigger an investigation and consider temporary containment."
    },

    "Critical": {
        "action": "Immediate Response",
        "description": "Trigger immediate incident response and prioritize containment."
    }
}

print("Response mapping created.\n")

for severity, response in RESPONSE_MAPPING.items():
    print(f"{severity}:")
    print("Action:", response["action"])
    print("Description:", response["description"])
    print()

Response mapping created.

Low:
Action: Monitor
Description: Continue monitoring the traffic and record the event.

Medium:
Action: Alert
Description: Generate an alert and increase monitoring.

High:
Action: Investigate
Description: Trigger an investigation and consider temporary containment.

Critical:
Action: Immediate Response
Description: Trigger immediate incident response and prioritize containment.



## Step 6.4 — Threat Response System

In [8]:
class ThreatResponseSystem:

    def __init__(
        self,
        model,
        target_encoder,
        scaler,
        severity_mapping,
        response_mapping
    ):
        self.model = model
        self.target_encoder = target_encoder
        self.scaler = scaler
        self.severity_mapping = severity_mapping
        self.response_mapping = response_mapping

    def decode_prediction(self, prediction):

        try:
            decoded = self.target_encoder.inverse_transform(
                np.asarray(prediction).reshape(-1, 1)
            )

            return decoded.ravel()[0]

        except Exception:

            try:
                return self.target_encoder.inverse_transform(
                    np.asarray(prediction).reshape(-1)
                )[0]

            except Exception:
                return str(prediction)

    def get_severity(self, attack):

        return self.severity_mapping.get(
            attack,
            "Medium"
        )

    def get_response(self, severity):

        return self.response_mapping.get(
            severity,
            {
                "action": "Monitor",
                "description": "Continue monitoring the event."
            }
        )

    def predict(self, X):

        predictions = self.model.predict(X)

        results = []

        for prediction in predictions:

            attack = self.decode_prediction([prediction])

            severity = self.get_severity(attack)

            response = self.get_response(severity)

            results.append({
                "attack": attack,
                "severity": severity,
                "response": response["action"],
                "description": response["description"]
            })

        return pd.DataFrame(results)

Create system

In [9]:
sim_system = ThreatResponseSystem(
    model=xgb_model,
    target_encoder=target_encoder,
    scaler=scaler,
    severity_mapping=SEVERITY_MAPPING,
    response_mapping=RESPONSE_MAPPING
)

print("ThreatResponseSystem created successfully.")

ThreatResponseSystem created successfully.


Test Threat Detection and Response

In [10]:
SAMPLE_SIZE = 20

X_sample = X_test.iloc[:SAMPLE_SIZE]

response_results = sim_system.predict(X_sample)

print("Threat response simulation completed.")
print("\nResults:")
display(response_results)

Threat response simulation completed.

Results:


d:\smart-cybersecurity-ml\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:161: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
d:\smart-cybersecurity-ml\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:161: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
d:\smart-cybersecurity-ml\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:161: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
d:\smart-cybersecurity-ml\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:161: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Ple

,attack,severity,response,description
0,apache2,Medium,Alert,Generate an alert and increase monitoring.
1,normal,Low,Monitor,Continue monitoring the traffic and record the...
2,apache2,Medium,Alert,Generate an alert and increase monitoring.
3,apache2,Medium,Alert,Generate an alert and increase monitoring.
4,apache2,Medium,Alert,Generate an alert and increase monitoring.
5,normal,Low,Monitor,Continue monitoring the traffic and record the...
6,apache2,Medium,Alert,Generate an alert and increase monitoring.
7,apache2,Medium,Alert,Generate an alert and increase monitoring.
8,apache2,Medium,Alert,Generate an alert and increase monitoring.
9,normal,Low,Monitor,Continue monitoring the traffic and record the...


Severity Analysis

In [11]:
severity_counts = (
    response_results["severity"]
    .value_counts()
    .reindex(
        ["Low", "Medium", "High", "Critical"],
        fill_value=0
    )
)

print("Severity Distribution:")
print(severity_counts)

Severity Distribution:
severity
Low          8
Medium      12
High         0
Critical     0
Name: count, dtype: int64


Response distribution

In [12]:
response_counts = response_results["response"].value_counts()

print("Response Distribution:")
print(response_counts)

Response Distribution:
response
Alert      12
Monitor     8
Name: count, dtype: int64


## Step 6.7 — Real-Time Threat Detection Simulation

In [13]:
def simulate_real_time_detection(
    system,
    X_data,
    sample_size=10
):

    sample = X_data.iloc[:sample_size]

    results = system.predict(sample)

    print("=" * 70)
    print("REAL-TIME THREAT DETECTION SIMULATION")
    print("=" * 70)

    for index, row in results.iterrows():

        print(f"\nEvent {index + 1}")
        print("-" * 40)
        print("Detected Attack :", row["attack"])
        print("Severity         :", row["severity"])
        print("Recommended Action:", row["response"])
        print("Description      :", row["description"])

    return results

Run simulation

In [14]:
real_time_results = simulate_real_time_detection(
    sim_system,
    X_test,
    sample_size=10
)

REAL-TIME THREAT DETECTION SIMULATION

Event 1
----------------------------------------
Detected Attack : apache2
Severity         : Medium
Recommended Action: Alert
Description      : Generate an alert and increase monitoring.

Event 2
----------------------------------------
Detected Attack : normal
Severity         : Low
Recommended Action: Monitor
Description      : Continue monitoring the traffic and record the event.

Event 3
----------------------------------------
Detected Attack : apache2
Severity         : Medium
Recommended Action: Alert
Description      : Generate an alert and increase monitoring.

Event 4
----------------------------------------
Detected Attack : apache2
Severity         : Medium
Recommended Action: Alert
Description      : Generate an alert and increase monitoring.

Event 5
----------------------------------------
Detected Attack : apache2
Severity         : Medium
Recommended Action: Alert
Description      : Generate an alert and increase monitoring.

Ev

d:\smart-cybersecurity-ml\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:161: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
d:\smart-cybersecurity-ml\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:161: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
d:\smart-cybersecurity-ml\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:161: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
d:\smart-cybersecurity-ml\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:161: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Ple

Step 6.8 — Automated Response Report

Report generator

In [15]:
def generate_response_report(
    results,
    severity_mapping,
    response_mapping
):

    report = []

    report.append("=" * 80)
    report.append("PHASE 6 — AUTOMATED THREAT RESPONSE REPORT")
    report.append("=" * 80)

    report.append("\n1. SYSTEM OVERVIEW")
    report.append("-" * 80)
    report.append(
        "This report summarizes simulated intrusion detection, "
        "severity classification, and recommended response actions."
    )

    report.append("\n2. TOTAL EVENTS")
    report.append("-" * 80)
    report.append(f"Total simulated events: {len(results)}")

    report.append("\n3. ATTACK DISTRIBUTION")
    report.append("-" * 80)

    attack_counts = results["attack"].value_counts()

    for attack, count in attack_counts.items():
        report.append(f"{attack}: {count}")

    report.append("\n4. SEVERITY DISTRIBUTION")
    report.append("-" * 80)

    severity_counts = (
        results["severity"]
        .value_counts()
        .reindex(
            ["Low", "Medium", "High", "Critical"],
            fill_value=0
        )
    )

    for severity, count in severity_counts.items():
        report.append(f"{severity}: {count}")

    report.append("\n5. RESPONSE DISTRIBUTION")
    report.append("-" * 80)

    response_counts = results["response"].value_counts()

    for response, count in response_counts.items():
        report.append(f"{response}: {count}")

    report.append("\n6. RESPONSE POLICY")
    report.append("-" * 80)

    for severity, response in response_mapping.items():
        report.append(
            f"{severity} -> {response['action']}: "
            f"{response['description']}"
        )

    report.append("\n7. INCIDENT DETAILS")
    report.append("-" * 80)

    for index, row in results.iterrows():

        report.append(
            f"Event {index + 1}: "
            f"Attack={row['attack']}, "
            f"Severity={row['severity']}, "
            f"Response={row['response']}"
        )

    report.append("\n8. CONCLUSION")
    report.append("-" * 80)
    report.append(
        "The simulated threat response system successfully maps "
        "detected attack classes to severity levels and recommended "
        "response actions."
    )

    return "\n".join(report)

Generate report

In [16]:
response_report = generate_response_report(
    real_time_results,
    SEVERITY_MAPPING,
    RESPONSE_MAPPING
)

report_path = os.path.join(
    PHASE6_RESULTS,
    "phase6_response_report.txt"
)

with open(report_path, "w", encoding="utf-8") as file:
    file.write(response_report)

print("Response report generated successfully.")
print("Saved to:")
print(report_path)

Response report generated successfully.
Saved to:
D:\smart-cybersecurity-ml\results\phase6\phase6_response_report.txt


Save Complete Threat Response System

In [17]:
phase6_system = {
    "model": xgb_model,
    "target_encoder": target_encoder,
    "scaler": scaler,
    "severity_mapping": SEVERITY_MAPPING,
    "response_mapping": RESPONSE_MAPPING,
    "system": sim_system
}

system_path = os.path.join(
    PHASE6_RESULTS,
    "phase6_threat_response_system.pkl"
)

joblib.dump(
    phase6_system,
    system_path
)

print("Complete threat response system saved successfully.")
print("Saved to:")
print(system_path)

Complete threat response system saved successfully.
Saved to:
D:\smart-cybersecurity-ml\results\phase6\phase6_threat_response_system.pkl


Final Verification

In [18]:
print("=" * 70)
print("PHASE 6 — FINAL OUTPUT VERIFICATION")
print("=" * 70)

expected_files = [
    "phase6_response_report.txt",
    "phase6_threat_response_system.pkl"
]

for filename in expected_files:

    path = os.path.join(
        PHASE6_RESULTS,
        filename
    )

    print(
        f"{filename}:",
        "EXISTS" if os.path.exists(path) else "MISSING"
    )

print("\nPhase 6 results folder:")
print(PHASE6_RESULTS)

PHASE 6 — FINAL OUTPUT VERIFICATION
phase6_response_report.txt: EXISTS
phase6_threat_response_system.pkl: EXISTS

Phase 6 results folder:
D:\smart-cybersecurity-ml\results\phase6
